# Trabajo Practico 4

En este trabajo se usa el modelo insumo-producto de Leontief. La idea del modelo es representar como cada sector productivo necesita insumos de los otros sectores para poder producir.

Cada coeficiente `a_ij` de la matriz `A` indica cuanto producto del sector `i` se necesita para producir una unidad del sector `j`. Por eso, si `x` es el vector de produccion total, la produccion usada internamente por el sistema se escribe como:

$$
Ax
$$

Ademas de esa demanda intermedia, existe una demanda final externa al sistema, representada por el vector `y`. Entonces el modelo plantea:

$$
x = Ax + y
$$

Pasando `Ax` al primer miembro:

$$
x - Ax = y
$$

Sacando factor comun `x`:

$$
(I - A)x = y
$$

Finalmente, si la matriz `(I - A)` tiene inversa, la produccion total se calcula con:

$$
x = (I - A)^{-1} y
$$

La matriz `(I - A)^{-1}` se conoce como inversa de Leontief. Sus valores incluyen tanto el efecto directo como el efecto indirecto entre sectores. Cuando ya se conoce esa inversa, el efecto de un cambio en la demanda final se obtiene con:

$$
\Delta x = (I - A)^{-1} \Delta y
$$


In [1]:
# FUNCIONES AUXILIARES

def identidad(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]

def restar_matrices(A, B):
    filas = len(A)
    columnas = len(A[0])
    return [[A[i][j] - B[i][j] for j in range(columnas)] for i in range(filas)]

def inversa_gauss_jordan(M):
    n = len(M)
    I = identidad(n)
    aumentada = []

    for i in range(n):
        aumentada.append(M[i][:] + I[i][:])

    for col in range(n):
        pivote = col
        for fila in range(col + 1, n):
            if abs(aumentada[fila][col]) > abs(aumentada[pivote][col]):
                pivote = fila

        aumentada[col], aumentada[pivote] = aumentada[pivote], aumentada[col]

        divisor = aumentada[col][col]
        if abs(divisor) < 1e-12:
            raise ValueError('La matriz no tiene inversa')

        for j in range(2 * n):
            aumentada[col][j] = aumentada[col][j] / divisor

        for fila in range(n):
            if fila != col:
                factor = aumentada[fila][col]
                for j in range(2 * n):
                    aumentada[fila][j] = aumentada[fila][j] - factor * aumentada[col][j]

    return [fila[n:] for fila in aumentada]

def multiplicar_matriz_vector(M, v):
    resultado = []
    for i in range(len(M)):
        suma = 0
        for j in range(len(v)):
            suma = suma + M[i][j] * v[j]
        resultado.append(suma)
    return resultado

def produccion_total(A, y):
    n = len(A)
    matriz_leontief = restar_matrices(identidad(n), A)
    inversa_leontief = inversa_gauss_jordan(matriz_leontief)
    x = multiplicar_matriz_vector(inversa_leontief, y)
    return matriz_leontief, inversa_leontief, x

def imprimir_matriz(nombre, M, sectores=None, decimales=4):
    print(nombre)
    if sectores is not None:
        encabezado = ' ' * 14
        for sector in sectores:
            encabezado += f'{sector:>14}'
        print(encabezado)

    for i, fila in enumerate(M):
        etiqueta = sectores[i] if sectores is not None else ''
        texto = f'{etiqueta:<14}'
        for valor in fila:
            if abs(valor) < 1e-12:
                valor = 0
            texto += f'{valor:>14.{decimales}f}'
        print(texto)
    print()

def imprimir_vector(nombre, sectores, v, decimales=4):
    print(nombre)
    for sector, valor in zip(sectores, v):
        if abs(valor) < 1e-12:
            valor = 0
        print(f'{sector:<14}: {valor:.{decimales}f}')
    print()


## Punto 1

Se tienen dos sectores: Agricultura e Industria.

Datos:

$$
A =
\begin{pmatrix}
0.2 & 0.3\\
0.1 & 0.4
\end{pmatrix},\qquad
y =
\begin{pmatrix}
200\\
150
\end{pmatrix}
$$

Se calcula:

$$
x = (I - A)^{-1}y
$$


In [2]:
# PUNTO 1

sectores_1 = ['Agricultura', 'Industria']

A1 = [
    [0.2, 0.3],
    [0.1, 0.4]
]

y1 = [200, 150]

leontief_1, inversa_1, x1 = produccion_total(A1, y1)

imprimir_matriz('Matriz I - A:', leontief_1, sectores_1)
imprimir_matriz('Inversa de Leontief (I - A)^(-1):', inversa_1, sectores_1)
imprimir_vector('Vector de produccion total x:', sectores_1, x1)


Matriz I - A:
                 Agricultura     Industria
Agricultura           0.8000       -0.3000
Industria            -0.1000        0.6000

Inversa de Leontief (I - A)^(-1):
                 Agricultura     Industria
Agricultura           1.3333        0.6667
Industria             0.2222        1.7778

Vector de produccion total x:
Agricultura   : 366.6667
Industria     : 311.1111



## Punto 2

Se consideran tres sectores: Energia, Transporte y Manufactura.

La inversa de Leontief ya esta dada. En este caso no se pide la produccion total inicial, sino analizar que pasa si cambia la demanda final.

Para un aumento de 50 unidades en la demanda final de Transporte, el cambio se representa con:

$$
\Delta y =
\begin{pmatrix}
0\\
50\\
0
\end{pmatrix}
$$

Entonces se calcula el cambio en la produccion total:

$$
\Delta x = (I - A)^{-1}\Delta y
$$

El resultado muestra cuanto debe aumentar la produccion de cada sector para cubrir ese aumento de demanda, considerando las relaciones directas e indirectas entre sectores.


In [3]:
# PUNTO 2

sectores_2 = ['Energia', 'Transporte', 'Manufactura']

inversa_2 = [
    [1.25, 0.10, 0.05],
    [0.20, 1.40, 0.10],
    [0.15, 0.20, 1.30]
]

aumento_y2 = [0, 50, 0]
aumento_x2 = multiplicar_matriz_vector(inversa_2, aumento_y2)

imprimir_matriz('Inversa de Leontief dada:', inversa_2, sectores_2)
imprimir_vector('Aumento de demanda final Delta y:', sectores_2, aumento_y2)
imprimir_vector('Efecto sobre la produccion Delta x:', sectores_2, aumento_x2)


Inversa de Leontief dada:
                     Energia    Transporte   Manufactura
Energia               1.2500        0.1000        0.0500
Transporte            0.2000        1.4000        0.1000
Manufactura           0.1500        0.2000        1.3000

Aumento de demanda final Delta y:
Energia       : 0.0000
Transporte    : 50.0000
Manufactura   : 0.0000

Efecto sobre la produccion Delta x:
Energia       : 5.0000
Transporte    : 70.0000
Manufactura   : 10.0000



## Punto 3

Se consideran cuatro sectores: Agricultura, Industria, Transporte y Servicios.

Datos:

$$
A =
\begin{pmatrix}
0.1 & 0.2 & 0.1 & 0.0\\
0.1 & 0.3 & 0.2 & 0.1\\
0.0 & 0.2 & 0.2 & 0.1\\
0.1 & 0.1 & 0.1 & 0.2
\end{pmatrix},\qquad
y =
\begin{pmatrix}
100\\
150\\
80\\
120
\end{pmatrix}
$$

Se calcula el vector de produccion total:

$$
x = (I - A)^{-1}y
$$


In [4]:
# PUNTO 3

sectores_3 = ['Agricultura', 'Industria', 'Transporte', 'Servicios']

A3 = [
    [0.1, 0.2, 0.1, 0.0],
    [0.1, 0.3, 0.2, 0.1],
    [0.0, 0.2, 0.2, 0.1],
    [0.1, 0.1, 0.1, 0.2]
]

y3 = [100, 150, 80, 120]

leontief_3, inversa_3, x3 = produccion_total(A3, y3)

imprimir_matriz('Matriz I - A:', leontief_3, sectores_3)
imprimir_matriz('Inversa de Leontief (I - A)^(-1):', inversa_3, sectores_3)
imprimir_vector('Vector de produccion total x:', sectores_3, x3)


Matriz I - A:
                 Agricultura     Industria    Transporte     Servicios
Agricultura           0.9000       -0.2000       -0.1000        0.0000
Industria            -0.1000        0.7000       -0.2000       -0.1000
Transporte            0.0000       -0.2000        0.8000       -0.1000
Servicios            -0.1000       -0.1000       -0.1000        0.8000

Inversa de Leontief (I - A)^(-1):
                 Agricultura     Industria    Transporte     Servicios
Agricultura           1.1673        0.4205        0.2617        0.0853
Industria             0.2146        1.6642        0.4763        0.2676
Transporte            0.0764        0.4557        1.4025        0.2323
Servicios             0.1823        0.3176        0.2676        1.3231

Vector de produccion total x:
Agricultura   : 210.9674
Industria     : 341.3114
Transporte    : 216.0835
Servicios     : 246.0453



## Punto 4

Se consideran cinco sectores: Agricultura, Industria, Energia, Transporte y Servicios.

La inversa de Leontief ya esta dada. Igual que en el punto 2, se analiza un cambio en la demanda final.

Como el aumento es de 100 unidades en Energia, el vector de cambio de demanda final queda:

$$
\Delta y =
\begin{pmatrix}
0\\
0\\
100\\
0\\
0
\end{pmatrix}
$$

Luego se multiplica por la inversa de Leontief para obtener el cambio en la produccion total:

$$
\Delta x = (I - A)^{-1}\Delta y
$$

Asi se obtiene el aumento requerido en cada sector, incluyendo el aumento directo en Energia y los aumentos indirectos en los demas sectores.


In [5]:
# PUNTO 4

sectores_4 = ['Agricultura', 'Industria', 'Energia', 'Transporte', 'Servicios']

inversa_4 = [
    [1.20, 0.10, 0.05, 0.00, 0.02],
    [0.15, 1.30, 0.10, 0.05, 0.00],
    [0.05, 0.10, 1.25, 0.10, 0.05],
    [0.10, 0.15, 0.10, 1.40, 0.10],
    [0.05, 0.05, 0.05, 0.10, 1.20]
]

aumento_y4 = [0, 0, 100, 0, 0]
aumento_x4 = multiplicar_matriz_vector(inversa_4, aumento_y4)

imprimir_matriz('Inversa de Leontief dada:', inversa_4, sectores_4)
imprimir_vector('Aumento de demanda final Delta y:', sectores_4, aumento_y4)
imprimir_vector('Efecto sobre la produccion Delta x:', sectores_4, aumento_x4)


Inversa de Leontief dada:
                 Agricultura     Industria       Energia    Transporte     Servicios
Agricultura           1.2000        0.1000        0.0500        0.0000        0.0200
Industria             0.1500        1.3000        0.1000        0.0500        0.0000
Energia               0.0500        0.1000        1.2500        0.1000        0.0500
Transporte            0.1000        0.1500        0.1000        1.4000        0.1000
Servicios             0.0500        0.0500        0.0500        0.1000        1.2000

Aumento de demanda final Delta y:
Agricultura   : 0.0000
Industria     : 0.0000
Energia       : 100.0000
Transporte    : 0.0000
Servicios     : 0.0000

Efecto sobre la produccion Delta x:
Agricultura   : 5.0000
Industria     : 10.0000
Energia       : 125.0000
Transporte    : 10.0000
Servicios     : 5.0000



## Resultados principales

- Punto 1: para satisfacer la demanda final se necesitan aproximadamente 366.67 unidades de Agricultura y 311.11 unidades de Industria.
- Punto 2: un aumento de 50 unidades en Transporte genera aumentos de 5 unidades en Energia, 70 en Transporte y 10 en Manufactura.
- Punto 3: el vector de produccion total es aproximadamente `(210.97, 341.31, 216.08, 246.05)`.
- Punto 4: un aumento de 100 unidades en Energia genera aumentos de 5 en Agricultura, 10 en Industria, 125 en Energia, 10 en Transporte y 5 en Servicios.
